In [35]:
catalog_id = 8
numpages = 17
catalog_url='https://nwmissouri.catalog.acalog.com/content.php?catoid={}&navoid=227&expand=1&cpage={}'

In [41]:
import requests
pagenum=1
print(catalog_url.format(catalog_id, pagenum))
page=requests.get(catalog_url.format(catalog_id, pagenum))

https://nwmissouri.catalog.acalog.com/content.php?catoid=8&navoid=227&expand=1&cpage=1


In [37]:
import pickle
with open("cat2627.pkl", 'wb') as f:
    pickle.dump(page, f)

In [38]:
with open("cat2627.pkl", 'rb') as f:
    landing = pickle.load(f)

In [39]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(landing.text, 'html5lib')

In [40]:
print(soup)

<html><head></head><body></body></html>


In [46]:
print(page.headers)

{'Date': 'Fri, 11 Sep 2026 19:00:51 GMT', 'Content-Type': 'text/html; charset=UTF-8', 'Content-Length': '0', 'Connection': 'keep-alive', 'x-amzn-waf-action': 'challenge', 'Cache-Control': 'no-store, max-age=0', 'Access-Control-Allow-Origin': '*', 'Access-Control-Max-Age': '86400', 'Access-Control-Allow-Methods': 'OPTIONS,GET,POST', 'Access-Control-Expose-Headers': 'x-amzn-waf-action', 'Server': 'director', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains'}


In [45]:
print(page.status_code)

202


In [65]:
from selenium import webdriver

browser = webdriver.Firefox()
browser.get('https://nwmissouri.catalog.acalog.com/content.php?catoid=8&navoid=227')

In [113]:
from selenium.webdriver.common.by import By
links = browser.find_elements(By.TAG_NAME, 'a')

In [114]:
courselinks = [l for l in links if l.get_attribute('href') and 'preview_course' in l.get_attribute('href')]

In [115]:
print(courselinks[0].text)

AGRI 03536 - Soil Fertility


In [116]:
for l in courselinks:
    l.click()

In [117]:
course_tds = browser.find_elements(By.CLASS_NAME, 'coursepadding')

In [118]:
print(len(course_tds))

100


In [72]:
dir(course_tds[0])

['__abstractmethods__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_execute',
 '_id',
 '_parent',
 '_upload',
 'accessible_name',
 'aria_role',
 'clear',
 'click',
 'find_element',
 'find_elements',
 'get_attribute',
 'get_dom_attribute',
 'get_property',
 'id',
 'is_displayed',
 'is_enabled',
 'is_selected',
 'location',
 'location_once_scrolled_into_view',
 'parent',
 'rect',
 'screenshot',
 'screenshot_as_base64',
 'screenshot_as_png',
 'send_keys',
 'shadow_root',
 'size',
 'submit',
 'tag_name',
 'text',
 'value_of_css_property']

In [73]:
course_tds[0].text

'ARHU 72101 - Freshman Seminar\nCredit Hours: 1'

In [74]:
n =course_tds[0].find_element(By.TAG_NAME,'h3')

In [77]:
print(course_tds[0][0])

TypeError: 'WebElement' object is not subscriptable

In [119]:
ctexts = [c.text for c in course_tds]

In [120]:
print(ctexts[0])

[Print Course (opens a new window)]
AGRI 03536 - Soil Fertility
Credit Hours: 3
Principles of soil productivity and nutrients required for crop growth; fertilizer sources and nutrient reactions in soil; methods of fertilizer nutrient placement in major tillage systems; interpretation of soil test and plant analyses for determining crop nutrient requirements.

Prerequisite(s): AGRI 03234 
Typically Offered: SP


In [86]:
import re
lines = ctexts[0].split('\n')
cinfo = re.compile('^([A-Z]{3}[A-Z]?) ([0-9]{5}) - (.*)$')
m= cinfo.match(lines[1])
m.groups()

AttributeError: 'NoneType' object has no attribute 'groups'

In [104]:
cinfo = re.compile('^([A-Z]{3}[A-Z]?) ([0-9]{5}) - (.*)$')
cid = re.compile('([0-9]{5})')

def parse_prereq_tree(prereqstr):
    # assume disjunctive normal form???
    clauses = [x.strip() for x in prereqstr.replace('.','').split(' or ')]
    cgroups = [[x.strip() for x in c.split(' and ')] for c in clauses]
    return cgroups

def parse_course(ct):
    lines = ct.split('\n')
    cnum = None
    prs = []
    for l in lines:
        if cinfo.match(l):
            cnum = cinfo.match(l).groups()[1]
        elif l.strip().startswith('Prere'):
            m = cid.findall(l.split(':')[1])
            if m:
                prs = m
                
            #val.append(parse_prereq_tree(l.split(':')[1].strip()))
    return {cnum: prs}
mapping = [parse_course(x) for x in ctexts]
print(mapping)

[{'72101': []}, {'71200': []}, {'71301': []}, {'71380': []}, {'51201': []}, {'51202': ['51201']}, {'51301': ['51202']}, {'51302': ['51201', '17114']}, {'51303': ['51202']}, {'51304': ['51202']}, {'51306': ['51202']}, {'51307': ['51306']}, {'51308': ['51306', '44130']}, {'51401': ['51306']}, {'51402': ['51301']}, {'51403': ['51307']}, {'51404': ['51307']}, {'51405': []}, {'51408': ['51202', '54313', '53324', '55330']}, {'51409': []}, {'03100': []}, {'03199': []}, {'03200': []}, {'03300': []}, {'03320': []}, {'03354': []}, {'03396': []}, {'03400': []}, {'03466': []}, {'03500': []}, {'03102': []}, {'03301': ['03102']}, {'03302': ['03102']}, {'03304': ['03102']}, {'03305': ['03102', '52151']}, {'03307': ['03304', '51201']}, {'03308': ['17118']}, {'03309': []}, {'03404': []}, {'03407': ['03304']}, {'03408': ['03304']}, {'03409': []}, {'03502': ['03102', '03302']}, {'03503': []}, {'03504': ['03309']}, {'03505': ['03102']}, {'03508': ['03102']}, {'03509': []}, {'03598': ['17114', '17610', '44

In [105]:
curr=browser.find_element(By.CSS_SELECTOR, '[aria-current="page"]')

In [106]:
print(curr.text)

1


In [109]:
nextlink = browser.find_element(By.CSS_SELECTOR,'[aria-label="Page {}"]'.format(int(curr.text)+1))

In [111]:
nextlink.click()

In [121]:
mapping.extend([parse_course(x) for x in ctexts])

In [122]:
print(mapping)

[{'72101': []}, {'71200': []}, {'71301': []}, {'71380': []}, {'51201': []}, {'51202': ['51201']}, {'51301': ['51202']}, {'51302': ['51201', '17114']}, {'51303': ['51202']}, {'51304': ['51202']}, {'51306': ['51202']}, {'51307': ['51306']}, {'51308': ['51306', '44130']}, {'51401': ['51306']}, {'51402': ['51301']}, {'51403': ['51307']}, {'51404': ['51307']}, {'51405': []}, {'51408': ['51202', '54313', '53324', '55330']}, {'51409': []}, {'03100': []}, {'03199': []}, {'03200': []}, {'03300': []}, {'03320': []}, {'03354': []}, {'03396': []}, {'03400': []}, {'03466': []}, {'03500': []}, {'03102': []}, {'03301': ['03102']}, {'03302': ['03102']}, {'03304': ['03102']}, {'03305': ['03102', '52151']}, {'03307': ['03304', '51201']}, {'03308': ['17118']}, {'03309': []}, {'03404': []}, {'03407': ['03304']}, {'03408': ['03304']}, {'03409': []}, {'03502': ['03102', '03302']}, {'03503': []}, {'03504': ['03309']}, {'03505': ['03102']}, {'03508': ['03102']}, {'03509': []}, {'03598': ['17114', '17610', '44

In [150]:
curr=browser.find_element(By.CSS_SELECTOR, '[aria-current="page"]')
nextlink = browser.find_element(By.CSS_SELECTOR,'[aria-label="Page {}"]'.format(int(curr.text)+1))
nextlink.click()
links = browser.find_elements(By.TAG_NAME, 'a')
courselinks = [l for l in links if l.get_attribute('href') and 'preview_course' in l.get_attribute('href')]
for l in courselinks:
    l.click()
course_tds = browser.find_elements(By.CLASS_NAME, 'coursepadding')
ctexts = [c.text for c in course_tds]
mapping.extend([parse_course(x) for x in ctexts])

In [155]:
import json
with open('2026-2027/allprereqs.json', 'w') as f:
    json.dump(cmap, f)

In [152]:
cmap = {}
for c in mapping:
    cmap.update(c)

In [154]:
print(len(cmap))

630


In [156]:
print(cmap)

{'72101': [], '71200': [], '71301': [], '71380': [], '51201': [], '51202': ['51201'], '51301': ['51202'], '51302': ['51201', '17114'], '51303': ['51202'], '51304': ['51202'], '51306': ['51202'], '51307': ['51306'], '51308': ['51306', '44130'], '51401': ['51306'], '51402': ['51301'], '51403': ['51307'], '51404': ['51307'], '51405': [], '51408': ['51202', '54313', '53324', '55330'], '51409': [], '03100': [], '03199': [], '03200': [], '03300': [], '03320': [], '03354': [], '03396': [], '03400': [], '03466': [], '03500': [], '03102': [], '03301': ['03102'], '03302': ['03102'], '03304': ['03102'], '03305': ['03102', '52151'], '03307': ['03304', '51201'], '03308': ['17118'], '03309': [], '03404': [], '03407': ['03304'], '03408': ['03304'], '03409': [], '03502': ['03102', '03302'], '03503': [], '03504': ['03309'], '03505': ['03102'], '03508': ['03102'], '03509': [], '03598': ['17114', '17610', '44130', '03308'], '03421': [], '03422': [], '03429': [], '03524': [], '03525': [], '03527': [], '03

In [157]:
print(cmap['44242'])

KeyError: '44242'

In [159]:
print(len(mapping))

630
